# Самостоятельная работа. Многоклассовая классификация

Цель работы — обучить полносвязную модель через `nn.Sequential` и оценить её ответы на отложенных данных.
Циклы обучения, валидации и тестирования нужно написать самостоятельно.

## Набор данных

Таблица `df` содержит 1500 искусственных наблюдений.
Столбцы `x1` и `x2` — координаты объекта.
Столбец `target` — номер одного из трёх классов.
Модель должна определять класс по двум координатам.

## Импорт

In [1]:
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

## Данные

In [2]:
rng = np.random.default_rng(21)
labels = np.repeat([0, 1, 2], 500)
angle = rng.uniform(0, 2 * np.pi, len(labels))
radius = 0.7 + 0.9 * labels + rng.normal(0, 0.18, len(labels))

df = pd.DataFrame({
    "x1": radius * np.cos(angle),
    "x2": radius * np.sin(angle),
    "target": labels,
})
df = df.sample(frac=1, random_state=21).reset_index(drop=True)

## 1. Предпросмотр данных

Выведите несколько строк, размер таблицы, типы столбцов и число пропусков.
Покажите число объектов каждого класса.
Постройте диаграмму рассеяния `x1` и `x2` с отдельным обозначением каждого класса.
Добавьте легенду и одинаковый масштаб координатных осей.
Достаточно таблиц и графика, письменный вывод не требуется.

## 2. Подготовка данных, Dataset и DataLoader

### 2.1. Разделение и масштабирование

Выделите признаки `x1`, `x2` и целевой столбец `target`.
Все дальнейшие данные берите только из `df`.
Разделите строки на `train`, `valid` и `test` в пропорции 60/20/20.
Зафиксируйте `random_state`.

При обоих разделениях сохраните доли классов с помощью `stratify`.

Сохраните исходные координаты и индексы строк для итоговых графиков.
Масштабируйте признаки по статистикам `train`.
К `valid` и `test` примените те же параметры.
Целевой столбец не масштабируйте.

Подсказка. Сначала отделите 20% строк для теста.
Затем выделите четверть оставшихся строк для валидации.
У `StandardScaler` используйте `fit` только для обучающих признаков.

### 2.2. Собственный Dataset

Напишите класс с методами `__init__`, `__len__` и `__getitem__`.
В конструкторе сохраните признаки как тензор `float32`, а метки как тензор `long`.
Метод `__getitem__` должен возвращать одну пару «признаки, метка».
Создайте отдельные объекты для `train`, `valid` и `test`.

Подсказка. Для всей части признаки имеют форму `(N, 2)`, а метки `(N,)`.
Одна метка — целое число от 0 до 2.
Не добавляйте к меткам ось длины один.

### 2.3. Загрузчики

Создайте `DataLoader` для каждой части.
Размер пакета выберите сами.
У обучающего загрузчика включите перемешивание, у двух остальных отключите.
Получите один пакет и выведите формы его тензоров.

Подсказка. Для итоговой оценки на `train` удобно создать ещё один загрузчик без перемешивания.
Он будет использовать тот же обучающий `Dataset`.

## 3. Функция потерь

Выберите функцию потерь для трёх взаимоисключающих классов.
Создайте объект `loss_function`.

Подсказка. Для такого представления меток подходит `nn.CrossEntropyLoss`.
Ей передают логиты формы `(B, 3)` и целочисленные метки формы `(B,)`.
Используйте среднюю потерю по пакету без весов классов.

## 4. Архитектура модели

Создайте полносвязную сеть через `nn.Sequential`.
Подайте на вход два признака и получите три логита на выходе.
Число скрытых слоёв, их ширину и функции активации выберите сами.
Выведите модель и форму её ответа на один пакет.

Подсказка. Перед `CrossEntropyLoss` не добавляйте `Softmax`.
Вероятности понадобятся позже для визуализации.
Признаки и параметры модели должны находиться на одном устройстве.

## 5. Оптимизатор

Выберите оптимизатор и задайте шаг обучения.
Передайте ему параметры своей модели.

Подсказка. Можно начать с Adam или SGD.
Сначала разместите модель на выбранном устройстве, затем создайте оптимизатор.
Для другой архитектуры потребуется новый оптимизатор.

## 6. Цикл обучения и валидации

Напишите цикл по эпохам и вложенный цикл по пакетам `train`.
Для каждого пакета очистите градиенты, вычислите логиты и потерю, выполните обратный проход и шаг оптимизатора.
После каждой эпохи проведите отдельный проход по `valid` без обновления весов.

Сохраняйте среднюю потерю и accuracy на `train` и `valid` для каждой эпохи.
При уменьшении потери на `valid` копируйте состояние модели.
После обучения восстановите лучшие сохранённые параметры.
Архитектуру и настройки выбирайте по `valid`, не используя `test`.

Подсказка. Перед обучающим проходом вызывайте `train()`.
Для валидации используйте `eval()` вместе с `torch.no_grad()`.
Метки получайте через `argmax` по оси классов.
Для accuracy делите общее число верных ответов на число объектов.
Потери пакетов усредняйте с учётом их размеров.
Копию весов удобно хранить через `deepcopy(model.state_dict())`.

## 7. Цикл тестирования

Напишите отдельный цикл по тестовому загрузчику.
Используйте модель с восстановленными параметрами.
Соберите логиты, правильные метки и предсказанные классы для всех объектов.
Сохраните среднюю тестовую потерю.

Подсказка. Здесь нужны `eval()` и `torch.no_grad()`.
Оптимизатор, `backward()` и обновление весов в этом цикле не используются.
Собирайте предсказания и ответы вместе, чтобы сохранить соответствие строк.

## 8. Метрики и история обучения

### 8.1. Итоговые метрики

Получите ответы сохранённой модели на `train` без обновления весов.
Рассчитайте accuracy и macro-F1 отдельно для `train` и `test`.
Соберите результаты в небольшую таблицу.

Подсказка. Macro-F1 усредняет F1 по трём классам с одинаковым весом.
Для расчёта используйте `average="macro"`.
Метрики сохранённой модели вычисляйте заново, а не берите из последней строки истории.

### 8.2. Графики по эпохам

Постройте график потерь на `train` и `valid`.
Отдельно постройте график accuracy на этих частях.
Отметьте эпоху, параметры которой сохранились.
Добавьте подписи осей и легенду.

Подсказка. Накопленные при обучении значения относятся к разным состояниям весов внутри эпохи.
Для сравнения метрик при одном состоянии модели можно делать дополнительный проход по `train` после эпохи.
Тестовые метрики по эпохам не вычисляйте.

## 9. Визуализация ответов модели

### 9.1. Классы на плоскости

Постройте два графика рассеяния для одних и тех же тестовых объектов.
На первом обозначьте правильные классы, на втором предсказанные.
Сохраните одинаковые цвета классов, диапазоны осей и масштаб координат.
На втором графике отдельно отметьте ошибочно классифицированные точки.

### 9.2. Матрица ошибок

Постройте матрицу ошибок по тестовым меткам.
Покажите все три класса, включая классы без ошибок.
Подпишите оси и значения в ячейках.

Подсказка. У `ConfusionMatrixDisplay.from_predictions` первым аргументом идут правильные метки.
Строки матрицы отвечают правильным классам, столбцы предсказанным.

### 9.3. Линии уровня вероятности

Создайте равномерную сетку по двум признакам в пределах обучающих данных.
Получите логиты для всех точек и преобразуйте их в вероятности по оси классов.
Выберите один класс и постройте линии его вероятности для уровней 0.2, 0.5 и 0.8.
Добавьте тестовые точки и подпишите линии.

Подсказка. Используйте `meshgrid`, затем объедините координаты в таблицу с двумя столбцами.
Примените масштабирование, подготовленное на `train`.
Вероятности выбранного класса верните к форме сетки через `reshape`.
Линия вероятности 0.5 у одного класса не описывает все границы многоклассовой модели.

### 9.4. Области предсказываемых классов

По логитам на той же сетке определите предсказываемый класс в каждой точке.
Покажите области трёх классов через `contourf`.
Поверх нанесите тестовые объекты с обозначением их правильного класса.
Сохраните цвета классов из предыдущих графиков.

Подсказка. Для выбора класса используйте `argmax` по оси классов.
Задайте границы цветовых уровней между целочисленными метками.
Не интерпретируйте номера классов как непрерывную величину.